# Section 3 — Quantization: FP16 vs 4-bit NF4

Model: **Qwen/Qwen2.5-1.5B-Instruct** (fits on T4 for both FP16 and 4-bit — apples-to-apples comparison on free-tier hardware).

**Runtime:** Runtime → Change runtime type → **T4 GPU** (required).

Measures:
- VRAM footprint
- Throughput (tokens/sec)
- Qualitative output on 5 fixed prompts (same prompts, both versions)

In [ ]:
!pip install -q torch transformers accelerate bitsandbytes

In [ ]:
import torch, time, gc
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

assert torch.cuda.is_available(), 'No GPU! Switch runtime to T4 GPU.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_TOKENS = 150

PROMPTS = [
    'Explain what insurance underwriting means in simple terms.',
    'Write a short Python function that calculates factorial recursively.',
    'What are 3 key differences between supervised and unsupervised learning?',
    'List 4 health benefits of the Mediterranean diet.',
    "Translate to French: 'The weather is nice today, let\\'s go for a walk.'",
]

def vram_mb():
    return torch.cuda.memory_allocated() / 1024**2

def gen_and_time(model, tok, prompt):
    msgs = [{'role': 'user', 'content': prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors='pt').to(model.device)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_TOKENS, do_sample=False)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    new_ids = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(new_ids, skip_special_tokens=True), len(new_ids), elapsed

def bench(model, tok, label):
    print(f'\n--- {label} ---')
    results = []
    for i, p in enumerate(PROMPTS, 1):
        text, n, dt = gen_and_time(model, tok, p)
        tps = n/dt if dt else 0
        print(f'  prompt {i}: {n} tokens, {dt:.2f}s, {tps:.1f} tok/s')
        results.append({'prompt': p, 'output': text, 'tokens': n, 'time': dt, 'tok_s': tps})
    return results

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
print('tokenizer ready')

In [ ]:
# ---- FP16 baseline ----
print('Loading FP16...')
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
model_fp16 = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16,
    device_map='auto', trust_remote_code=True)
fp16_vram = vram_mb()
print(f'  VRAM: {fp16_vram:.0f} MB')
fp16_results = bench(model_fp16, tokenizer, 'FP16')
del model_fp16; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- 4-bit NF4 (bitsandbytes) ----
print('Loading 4-bit NF4...')
torch.cuda.reset_peak_memory_stats()
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model_q4 = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb,
    device_map='auto', trust_remote_code=True)
q4_vram = vram_mb()
print(f'  VRAM: {q4_vram:.0f} MB')
q4_results = bench(model_q4, tokenizer, '4-bit NF4')
del model_q4; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---- Side-by-side output comparison (all 5 prompts) ----
print('=' * 70)
print('  QUALITATIVE OUTPUT COMPARISON — 5 fixed prompts, both versions')
print('=' * 70)
for i, (a, b) in enumerate(zip(fp16_results, q4_results), 1):
    print(f'\n--- prompt {i}: {a["prompt"]}')
    print('\n  [FP16]')
    print('  ' + a['output'].strip().replace('\n', '\n  '))
    print('\n  [4-bit NF4]')
    print('  ' + b['output'].strip().replace('\n', '\n  '))

In [ ]:
# ---- Trade-off summary table ----
fp16_avg = sum(r['tok_s'] for r in fp16_results) / len(fp16_results)
q4_avg = sum(r['tok_s'] for r in q4_results) / len(q4_results)
mem_save = (1 - q4_vram/fp16_vram) * 100 if fp16_vram else 0
speed_delta = (q4_avg/fp16_avg - 1) * 100 if fp16_avg else 0

print('=' * 70)
print('  TRADE-OFF SUMMARY — precision vs size vs speed vs quality')
print('=' * 70)
print(f'\n{"Metric":<28}{"FP16":>14}{"4-bit NF4":>14}{"Delta":>14}')
print('-' * 70)
print(f'{"Precision":<28}{"FP16":>14}{"NF4 (4-bit)":>14}{"":>14}')
print(f'{"VRAM (MB)":<28}{fp16_vram:>14.0f}{q4_vram:>14.0f}{-mem_save:>+13.0f}%')
print(f'{"Throughput (tok/s avg)":<28}{fp16_avg:>14.1f}{q4_avg:>14.1f}{speed_delta:>+13.1f}%')
print(f'{"Quality (visual eval)":<28}{"baseline":>14}{"~near-identical":>14}{"":>14}')
print('\nSee NOTES.md for a discussion of when GPTQ/AWQ or GGUF beats bitsandbytes.')